# Inpaint Backgrounds

## Setup

In [1]:
import sys
from pathlib import Path
from PIL import Image
import random

# Add the project root to sys.path to allow imports from avm package
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from lib.utils import load_info_records
from lib.inpaint import (
    MODELS,
    detect_device,
    get_prompt,
    build_model,
    inpaint_background,
    get_model_config_defaults
)

## Constants

In [2]:
IMAGES_DIR = '../data/bg1k_imgs/'  # Contains images named <ID>.png e.g., 0.png, 1.png, ...
MASKS_DIR = '../data/bg1k_masks/'  # Contains masks named <ID>_mask.png e.g., 0_mask.png
INFO_TXT_PATH = '../data/bg1k_info.txt'  # Each line: <image name>\t<category>
MODEL_ID = MODELS['flux-finetuned-fixed-prompt']
LORA_PATH = PROJECT_ROOT / 'models/flux1_fill.pytorch_lora_weights.safetensors'  # Path to the LoRA weights
OUT_IMAGES_DIR = f'../data/bg1k_out_imgs/{MODEL_ID}/'  # Stores output images named <ID>.png

SEED = 42
PROMPT_OVERRIDE = ''  # If non-empty, overrides category prompt for all images

# Config overrides
CONFIG_OVERRIDES: dict[str, int | float] = {
    # Example overrides:
    # "num_inference_steps": 30,
}
MAX_RECORDS: int | None = 20  # None means no limit
OVERWRITE = False
DEVICE = None  # None -> auto detect via detect_device()

## Load Records

In [3]:
records = list(load_info_records(INFO_TXT_PATH))
if MAX_RECORDS is not None:
    records = random.Random(SEED).sample(records, min(MAX_RECORDS, len(records)))

print(f'Loaded {len(records)} records.')
print(f'Example records:')
for i, record in enumerate(records[:3]):
    print(f'{record}')

Loaded 20 records.
Example records:
{'image_id': '654', 'image_name': '654.png', 'category': 'Warm milk sterilization', 'mask_image_name': '654_mask.png'}
{'image_id': '114', 'image_name': '114.png', 'category': 'keyboard', 'mask_image_name': '114_mask.png'}
{'image_id': '25', 'image_name': '25.png', 'category': 'fabric sofa', 'mask_image_name': '25_mask.png'}


## Load Model

In [4]:
device = DEVICE or detect_device()
model = build_model(MODEL_ID, device, lora_path=str(LORA_PATH))
model

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

📦 Loading 1 LoRA file(s)...
🔧 Applying LoRA: flux1_fill.pytorch_lora_weights.safetensors (scale=1.0)
   ✅ Applied to 342 layers (684/684 keys matched)
✅ All LoRA weights applied successfully


## Generate

In [5]:
config = {
    **get_model_config_defaults(MODEL_ID),
    **CONFIG_OVERRIDES,
}
print(f'Config: {config}')

Config: {'guidance': 30.0, 'num_inference_steps': 28}


In [6]:
success = 0
skipped = 0
failed = 0

out_dir = Path(OUT_IMAGES_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

try:

    for idx, record in enumerate(records, start=1):
        image_name = record['image_name']
        mask_image_name = record['mask_image_name']
        category = record['category']
        image_path = Path(IMAGES_DIR) / image_name
        mask_image_path = Path(MASKS_DIR) / mask_image_name
        out_path = out_dir / image_name

        if out_path.exists() and not OVERWRITE:
            skipped += 1
            print(f'[{idx}/{len(records)}] Skipping existing output: {out_path.name}')
            continue

        category_prompt = get_prompt(category.strip())
        rendered_prompt = PROMPT_OVERRIDE.strip() or category_prompt

        print(f'[{idx}/{len(records)}] Processing {record["image_id"]}.png...')
        input_image = Image.open(image_path)
        mask_image = Image.open(mask_image_path)

        output_image = inpaint_background(
            model,
            input_image,
            mask_image,
            rendered_prompt,
            seed=SEED,
            config=config
        )

        output_image.save(out_path)
        success += 1
        print(f'[{idx}/{len(records)}] Saved: {out_path}')
except Exception as e:
    print(f'Error during processing: {e}')
finally:
    print(f'Finished processing records: {success} succeeded, {failed} failed, {skipped} skipped.')

[1/20] Skipping existing output: 654.png
[2/20] Processing 114.png...


100%|██████████| 28/28 [05:30<00:00, 11.80s/it]


[2/20] Saved: ../data/bg1k_out_imgs/flux-finetuned-fixed-prompt/114.png
[3/20] Processing 25.png...


100%|██████████| 28/28 [05:55<00:00, 12.70s/it]


[3/20] Saved: ../data/bg1k_out_imgs/flux-finetuned-fixed-prompt/25.png
[4/20] Processing 759.png...


...

[18/20] Saved: ../data/bg1k_out_imgs/flux-finetuned-fixed-prompt/32.png
[19/20] Processing 30.png...


100%|██████████| 28/28 [06:02<00:00, 12.96s/it]


[19/20] Saved: ../data/bg1k_out_imgs/flux-finetuned-fixed-prompt/30.png
[20/20] Processing 95.png...


100%|██████████| 28/28 [05:58<00:00, 12.82s/it]


[20/20] Saved: ../data/bg1k_out_imgs/flux-finetuned-fixed-prompt/95.png
Finished processing records: 19 succeeded, 0 failed, 1 skipped.
